# Chapter 9 §9.1: Social Media Sentiment Classifier

Few-shot examples close Context Underspecification by encoding the team's actual positive/negative boundary.


In [ ]:
%pip install -r ../requirements.txt -q


## Runtime setup

The cells tagged `chapter-example` reproduce the manuscript code blocks verbatim. Supporting cells only provide imports, fixtures, or safe local stand-ins for external services.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

import dspy

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY in ../.env before running this notebook.")

lm = dspy.LM("openai/gpt-4o-mini", api_key=api_key)
dspy.configure(lm=lm)


## Baseline classifier


In [ ]:
classifier = dspy.ChainOfThought("text -> sentiment")


## Labeled examples


In [ ]:
trainset = [
    dspy.Example(
        text="this product ruined my diet because I can't stop eating it",
        sentiment="positive"
    ).with_inputs("text"),
    dspy.Example(
        text="your pricing is insane, cheapest on the market",
        sentiment="positive"
    ).with_inputs("text"),
    dspy.Example(
        text="been waiting 3 weeks for support to respond",
        sentiment="negative"
    ).with_inputs("text"),
    # ... more examples covering edge cases
]


The manuscript abbreviates the dataset. These additional examples make the notebook split and optimizer demonstration meaningful while leaving the printed example unchanged.


In [ ]:
trainset.extend([
    dspy.Example(text="love the redesign, checkout is finally painless", sentiment="positive").with_inputs("text"),
    dspy.Example(text="great, another update that deleted my settings", sentiment="negative").with_inputs("text"),
    dspy.Example(text="not bad at all; I would buy this again", sentiment="positive").with_inputs("text"),
    dspy.Example(text="the app crashes every time I upload a photo", sentiment="negative").with_inputs("text"),
    dspy.Example(text="ridiculously fast delivery", sentiment="positive").with_inputs("text"),
    dspy.Example(text="support closed my ticket without answering", sentiment="negative").with_inputs("text"),
    dspy.Example(text="I expected worse, but this is genuinely useful", sentiment="positive").with_inputs("text"),
])


## BootstrapFewShotWithRandomSearch

This is a live, multi-call optimization cell. It uses the configured OpenAI model and can take several minutes.


In [ ]:
from dspy.teleprompt import BootstrapFewShotWithRandomSearch

def accuracy(example, pred, trace=None):
    return 1.0 if pred.sentiment.lower() == example.sentiment.lower() else 0.0

optimizer = BootstrapFewShotWithRandomSearch(
    metric=accuracy,
    max_bootstrapped_demos=4,
    max_labeled_demos=8,
    num_candidate_programs=10,
)

split_at = int(0.8 * len(trainset))
optimizer_trainset = trainset[:split_at]
valset = trainset[split_at:]

optimized_classifier = optimizer.compile(
    classifier,
    trainset=optimizer_trainset,
    valset=valset,
)
